# Student outcomes: supervised models and exploratory profiles

This notebook calls the same modules as the command line. It defaults to a
**fictitious 180-row fixture**; its scores have no empirical educational meaning.
Full-data outputs are recorded in [results/corrected_full/README.md](results/corrected_full/README.md).

The real dataset is not included; a full-data run needs an authorised local copy.
The original holdout was historically explored; a changed split does not make
it unseen. Academic context and credit are in [README.md](README.md), with
execution limits and test instructions there. Further methodological details
are in [technical notes](docs/technical-notes.md).


In [1]:
from pathlib import Path
import json
import pandas as pd
from main import run
from preprocessing import load_data, split_indices, prepare_classification, prepare_regression

data_path = Path("data/synthetic_students.csv")
output = Path("results/notebook_demo")
synthetic = True
folds = 2  # Use five folds for the full-data experiment.

## Shared rows, task-specific predictors

The outcome-stratified 80/20 split is shared by EDA and both supervised tasks. Classification includes second-semester information and is not an early-entry warning model. Regression excludes the six second-semester fields and the final outcome; the observation time of administrative predictors still needs confirmation.

In [2]:
frame = load_data(data_path)
train, holdout = split_indices(frame)
classification_data = prepare_classification(frame, train, holdout)
regression_data = prepare_regression(frame, train, holdout)
assert set(classification_data[0].index) == set(regression_data[0].index)
assert set(train).isdisjoint(holdout)
print(f"Rows: {len(frame)}, training: {len(train)}, holdout: {len(holdout)}")

Rows: 180, training: 144, holdout: 36


## Execute the three tasks

Encoding and scaling are fitted within each training fold. Model comparison uses the original seven candidates per task, and Gradient Boosting selection retains the original restricted five-configuration grid. The depth-4 classification comparison baseline is not in that grid. No holdout metric selects a model. PCA/K-means/Ward are exploratory full-data analyses; the k=3 interpretation is reported alongside the best silhouette k.

In [3]:
summary = run(data_path, output, folds=folds, synthetic=synthetic)

Rows: 180; train: 144; holdout: 36; folds: 2
Classification CV: LogisticRegression
Classification CV: LogReg_balanced
Classification CV: KNN
Classification CV: DecisionTree
Classification CV: RandomForest
Classification CV: RF_balanced
Classification CV: GradientBoosting
Classification grid: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Classification grid: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1}
Classification grid: {'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.1}
Classification grid: {'n_estimators': 150, 'max_depth': 5, 'learning_rate': 0.1}
Classification grid: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05}
Regression CV: Ridge
Regression CV: Lasso
Regression CV: ElasticNet
Regression CV: KNN
Regression CV: DecisionTree
Regression CV: RandomForest
Regression CV: GradientBoosting
Regression grid: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Regression grid: {'n_estimators': 100, 'max_depth': 5, 'learning_rate'

In [4]:
pd.read_csv(output / "regression_comparison.csv")

              model   r2_mean    r2_std  ...  rmse_std  mae_mean   mae_std
0  GradientBoosting  0.583502  0.043873  ...  0.131190  2.379944  0.028529
1      RandomForest  0.557219  0.018328  ...  0.036943  2.433837  0.000960
2             Lasso  0.512639  0.029656  ...  0.070104  2.524057  0.036356
3        ElasticNet  0.466863  0.025930  ...  0.052739  2.655164  0.029845
4      DecisionTree  0.266491  0.018168  ...  0.083091  2.919471  0.004243
5             Ridge  0.186872  0.097977  ...  0.214974  3.324572  0.165048
6               KNN  0.076249  0.010711  ...  0.064060  3.683231  0.050149

[7 rows x 7 columns]

## Interpret results within their scope

These notebook outputs are synthetic checks. For the corrected course-data run, consult `results/corrected_full/summary.json` and `run_metadata.json`, which record the actual data hash, sample size, seed, CV folds and package versions. Zero grades do not necessarily mean zero evaluations. Clusters and feature importance describe associations; they do not establish causes or validated student interventions.

In [5]:
metadata = json.loads((output / "run_metadata.json").read_text())
print({key: metadata[key] for key in ["synthetic", "run_rows", "cv_folds", "zero_grade_observations"]})

{'synthetic': True, 'run_rows': 180, 'cv_folds': 2, 'zero_grade_observations': {'zero_grades': 18, 'zero_grades_with_evaluations': 18}}
